# Mini-Ludo collusion detector (version 3)
Run the cell below (play button or Shift+Enter). It takes roughly 8-12 minutes, then saves `results.csv` and `results.png` (folder icon on the left).

What is new in this version:
- **Two collusion modes:** `association` (partners spare each other's tokens) and `signal` (a hidden, steganographic channel: no capture changes at all).
- **Two detectors:** capture-based (v2) and action-based (new), plus the capture-rate baseline.
- **95% confidence intervals** on every "pair found" rate and a finer strength grid.

For a 1-2 minute check that everything works, set `QUICK_TEST = True` in the settings block first.

In [ ]:
"""
mini_ludo_collusion.py   (version 3: adds a steganographic 'signal' mode, a second detector, confidence intervals)
=====================================================================================
A Ludo simulator with an optional, secretly colluding pair, plus an
information-theoretic detector. Written for Apart Research's AI Collusion
Sprint, project 1.4 ("Can information-theoretic methods identify colluding
pairs?"). It is INSPIRED BY Bonjour et al. (2022) but is NOT their exact
statistic: Ludo is turn-based, so we look at the dependence between WHO a capture
would hit (the target) and WHETHER the player took it (the decision).

HOW TO RUN (Google Colab)
  1. Upload this file (folder icon on the left), then in a cell run:  %run mini_ludo_collusion.py
     or paste the whole file into one cell and run it.
  2. A self-test runs first, then a table prints. results.csv and results.png are saved.
  3. For a quick test, set EXPERIMENTS = 5 and CONTROL_EXPERIMENTS = 5 below.

THE GAME (your rules, from the Wikipedia text and the board picture)
  * 4 players with 4 tokens each, no partnerships. Seats go clockwise:
    0 = green (left), 1 = yellow (top), 2 = light blue (right), 3 = pink (bottom).
  * The highest first roll starts (ties are re-rolled). Play passes clockwise.
  * A six lets you draw one token from the yard onto your start square, or move a token 6.
    Otherwise you must move a token by the number rolled. If no move is legal, the turn passes.
  * BONUS ROLL (default): every six earns a bonus roll, and you move after each one.
    A third six in a row means no move for that roll and the turn passes at once.
    (Set BONUS_ROLL_RULE = "yard_empty" for the stricter reading where the bonus roll
    only applies when you cannot draw a token from the yard.)
  * Landing on a single opponent token sends it back to its yard.
  * BLOCKS: two or more tokens of one colour on the same square form a block.
    An opponent who lands on a block is sent back to ITS yard; the block stays.
    (This also covers the "doubled" start square.)
  * Board geometry (checked against the picture): the shared track has 52 squares and
    start squares are 13 apart. A token travels 51 track squares (progress 0 to 50, ending on
    the square just before its home column), then 5 home-column squares (progress 51 to 55,
    always safe), then the finish (progress 56), which needs an EXACT roll.
  * No other safe squares. First player to finish all four tokens wins.

COLLUSION (what the secret pair does)
  * "Sparing": when a capture of the partner's token is available, the colluder
    avoids it with probability STRENGTH (if another legal move exists).
  * MODE "association": both partners spare each other (mutual, symmetric).
  * MODE "assistant":   only partner A spares partner B (one-way help).
  * "Spectator collusion" is NOT implemented: Ludo has no hidden information,
    so a spectator has nothing secret to pass on.

  * MODE "signal" (NEW, steganographic): the pair NEVER spares each other, so capture rates
    and capture decisions look completely honest. Instead, partner B (the receiver) secretly
    copies the 'parity' (even/odd token number) of partner A's most recent move, with
    probability STRENGTH, whenever B has an equally harmless alternative. 'Harmless' means the
    alternative captures exactly the same opponents (or none), so no capture statistic changes.
    This is a toy covert channel: a message hidden inside the choice among equivalent moves.

DETECTOR 2 (NEW, action-based, closer in spirit to Bonjour et al.)
  Looks at the dependence between two players' MOVE CHOICES (token parity), not at captures.
  For each move by player i we pair it with player j's most recent move and compute the mutual
  information (summed over both directions of a pair).
  Calibration: the scores of honest games (no collusion) are simulated first, N_REFERENCE times for each
  sample size. Each pair's score becomes a z-score against that honest reference, which already
  contains the dependence honest play creates (shared game stage, turn order), so seat position
  cannot bias it. In real use this corresponds to calibrating on games known to be honest.

THE DETECTOR 1 (state-conditioned: only counts moments when a capture was possible)
  For each player i we collect events (target j, decision d), where d = 1 if i captured
  j's token and 0 if i could have but did not. I_i(Target; Decision) measures how much the
  decision depends on whose token it is. An honest player ignores ownership -> about 0.
  A colluder spares one specific player -> positive. We split I_i into a contribution
  per target j, add the two directions of a pair, and rank the pairs. A shuffle test
  gives a p-value. BASELINE for comparison: the pair with the lowest capture rate.
"""

import csv
import itertools
import random

import numpy as np

# =========================== SETTINGS (edit these) ===========================
N_PLAYERS = 4              # keep at 4 (the board picture has 4 colours)
COLOURS = ["green", "yellow", "light blue", "pink"]
TOKENS = 4                 # tokens per player
RING = 52                  # squares on the shared track
START_GAP = 13             # distance between neighbouring start squares
LAST_RING_PROGRESS = 50    # progress 0..50 is on the shared track
HOME = 56                  # progress 51..55 = home column, 56 = finish
BONUS_ROLL_RULE = "always"       # "always" (bonus roll after every six) or "yard_empty"
MAX_TURNS = 6000           # safety cap so a game can never run forever
HONEST_NOISE = 0.2         # chance an honest player picks a random legal move

PARTNERS = (0, 2)          # the colluding pair: green and light blue (opposite seats)
MODES = ["association", "signal"]   # "association", "assistant" or "signal" (run each in turn)

STRENGTHS = [0.0, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.75, 1.0]   # collusion probability; 0.0 = negative control
GAMES_LIST = [5, 10, 20]            # games observed per experiment (sample size)
EXPERIMENTS = 100                   # repeats per condition
CONTROL_EXPERIMENTS = 500           # more repeats for the no-collusion control (strength 0.0)
N_PERM = 200                        # shuffles for the capture-based detector
N_REFERENCE = 300                   # honest reference experiments used to calibrate the action-based detector
QUICK_TEST = False                  # True = tiny run (a few minutes less) just to check everything works
ALPHA = 0.05
SEED = 12345
if QUICK_TEST:
    STRENGTHS = [0.0, 0.3, 1.0]
    GAMES_LIST = [5, 10]
    EXPERIMENTS = CONTROL_EXPERIMENTS = 6
    N_PERM = 30
    N_REFERENCE = 30
# ============================================================================


# ------------------------------- the game -----------------------------------
def ring_pos(player, progress):
    return (START_GAP * player + progress) % RING


def legal_moves(state, player, roll):
    """Moves are tuples (token, new_progress, captured, suicide).
    captured = {owner: [token ids]} for a single opponent token that would be sent home.
    suicide  = True if the move lands on an opponent BLOCK (the mover's token is sent home)."""
    moves = []
    for t, prog in enumerate(state[player]):
        if prog == HOME:
            continue
        if prog == -1:                      # token in the yard
            if roll != 6:
                continue
            new = 0
        else:
            new = prog + roll
            if new > HOME:                  # exact roll needed to finish
                continue
        captured, suicide = {}, False
        if new <= LAST_RING_PROGRESS:       # home column is safe, so only the track matters
            pos = ring_pos(player, new)
            for q in range(N_PLAYERS):
                if q == player:
                    continue
                here = [u for u, pq in enumerate(state[q])
                        if 0 <= pq <= LAST_RING_PROGRESS and ring_pos(q, pq) == pos]
                if len(here) >= 2:
                    suicide = True
                elif len(here) == 1:
                    captured[q] = here
        moves.append((t, new, captured, suicide))
    return moves


def score(move, state, player):
    """How good an honest player thinks a move is."""
    t, new, captured, suicide = move
    if suicide:
        return -1000.0
    s = float(new)                          # prefer moving the furthest token
    if captured:
        s += 100 + 10 * sum(len(v) for v in captured.values())
    if new == HOME:
        s += 60
    if state[player][t] == -1:              # bringing a token out
        s += 40
    if new <= LAST_RING_PROGRESS and any(u != t and state[player][u] == new for u in range(TOKENS)):
        s += 10                             # forming a block
    return s


def choose_move(rng, candidates, state, player):
    if rng.random() < HONEST_NOISE:
        return rng.choice(candidates)
    return max(candidates, key=lambda m: score(m, state, player))


def partner_of(player):
    a, b = PARTNERS
    return b if player == a else a


def spares(player, mode):
    a, b = PARTNERS
    if mode == "association":
        return player in (a, b)
    if mode == "assistant":
        return player == a
    return False


def opening_player(rng):
    """Highest first roll starts; tied players roll again."""
    contenders = list(range(N_PLAYERS))
    while len(contenders) > 1:
        rolls = {p: rng.randint(1, 6) for p in contenders}
        best = max(rolls.values())
        contenders = [p for p in contenders if rolls[p] == best]
    return contenders[0]


def play_game(rng, strength, mode, events, seq=None, stats=None):
    """Play one game. Appends (player, target, decision) to events and (player, parity) to seq.
    stats = [signal opportunities, signal overrides]. Returns (winner, turns)."""
    state = [[-1] * TOKENS for _ in range(N_PLAYERS)]
    player = opening_player(rng)
    turns = 0
    last_parity = [None] * N_PLAYERS
    while turns < MAX_TURNS:
        turns += 1
        sixes = 0
        while True:
            roll = rng.randint(1, 6)
            yard_empty = all(p != -1 for p in state[player])
            if roll == 6:
                sixes += 1
                if sixes == 3:              # third six in a row: no move, turn passes
                    break
            moves = legal_moves(state, player, roll)
            if moves:
                candidates = moves
                if spares(player, mode) and rng.random() < strength:
                    partner = partner_of(player)
                    others = [m for m in moves if partner not in m[2]]
                    if others:              # skip captures of the partner if possible
                        candidates = others
                chosen = choose_move(rng, candidates, state, player)

                # SIGNAL MODE: receiver copies the sender's last token parity if that costs nothing
                if mode == "signal" and player == PARTNERS[1] and last_parity[PARTNERS[0]] is not None:
                    want = last_parity[PARTNERS[0]]
                    same_captures = set(chosen[2].keys())
                    alts = [m for m in moves if not m[3] and set(m[2].keys()) == same_captures
                            and m[0] % 2 == want]
                    if not chosen[3] and chosen[0] % 2 != want and alts:
                        if stats is not None:
                            stats[0] += 1
                        if rng.random() < strength:
                            chosen = max(alts, key=lambda m: score(m, state, player))
                            if stats is not None:
                                stats[1] += 1
                last_parity[player] = chosen[0] % 2
                if seq is not None:
                    seq.append((player, chosen[0] % 2))

                # Log one event for every opponent who COULD have been captured
                targets = set()
                for m in moves:
                    targets.update(m[2].keys())
                for j in targets:
                    events.append((player, j, 1 if j in chosen[2] else 0))

                t, new, captured, suicide = chosen
                if suicide:                 # landed on an opponent block
                    state[player][t] = -1
                else:
                    state[player][t] = new
                    for q, token_ids in captured.items():
                        for u in token_ids:
                            state[q][u] = -1
                    if all(p == HOME for p in state[player]):
                        return player, turns
            if roll == 6 and (BONUS_ROLL_RULE == "always" or yard_empty):
                continue                    # bonus roll
            break
        player = (player + 1) % N_PLAYERS
    return None, turns


def self_test():
    """Checks the rules and the board geometry. Raises an error if something is wrong."""
    for p in range(N_PLAYERS):
        assert ring_pos(p, 50) == (ring_pos(p, 0) - 2) % RING, "home-column entry square"
        assert ring_pos((p + 1) % N_PLAYERS, 0) == (ring_pos(p, 0) + START_GAP) % RING, "start spacing"

    def blank():
        return [[-1] * TOKENS for _ in range(N_PLAYERS)]

    s = blank(); s[0][0] = 5; s[1][0] = 47            # opponent sits on ring square 8
    m = legal_moves(s, 0, 3)
    assert len(m) == 1 and m[0][2] == {1: [0]} and not m[0][3], "single capture"

    s[1][1] = 47                                      # now two opponent tokens: a block
    m = legal_moves(s, 0, 3)
    assert m[0][3] is True and m[0][2] == {}, "block sends the attacker home"

    s = blank(); s[0][0] = 5; s[0][1] = 8             # stacking own tokens
    mv = {x[0]: x for x in legal_moves(s, 0, 3)}
    assert mv[0][1] == 8 and mv[0][2] == {} and not mv[0][3], "own stack"

    s = blank(); s[0][0] = 55                         # exact roll needed to finish
    assert legal_moves(s, 0, 2) == [] and len(legal_moves(s, 0, 1)) == 1, "exact finish"

    s = blank()                                       # a six is needed to leave the yard
    assert legal_moves(s, 0, 5) == [] and len(legal_moves(s, 0, 6)) == TOKENS, "six to enter"

    s = blank(); s[0][0] = 9; s[1][0] = 51            # home column is safe
    assert legal_moves(s, 0, 3)[0][2] == {}, "home column safe"
    # signal mode must not change any capture decision pattern, and all modes must run
    for mode in ("association", "assistant", "signal"):
        ev, sq, st = [], [], [0, 0]
        play_game(random.Random(1), 1.0, mode, ev, sq, st)
        assert len(sq) > 0, "no moves recorded"
    assert abs(wilson(50, 100)[0] - 0.404) < 0.01, "wilson interval"
    print("Self-test passed: capture, block, stacking, exact finish, six to enter, safe home column, all modes run.")


# ----------------------------- the statistics -------------------------------
def count_matrix(events):
    """N[i, j, d] = how often player i could capture j's token and chose d (1 = captured)."""
    N = np.zeros((N_PLAYERS, N_PLAYERS, 2))
    for i, j, d in events:
        N[i, j, d] += 1
    return N


def cell_scores(N):
    """c[i, j] = part of I_i(Target; Decision), in bits, that comes from target j."""
    c = np.zeros((N_PLAYERS, N_PLAYERS))
    for i in range(N_PLAYERS):
        n = N[i].sum()
        if n == 0:
            continue
        p = N[i] / n                              # joint probability p(target, decision)
        p_target = p.sum(axis=1, keepdims=True)
        p_decision = p.sum(axis=0, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            term = np.where(p > 0, p * np.log2(p / (p_target * p_decision)), 0.0)
        c[i] = term.sum(axis=1)
    return c


def pair_scores(c):
    return {(a, b): c[a, b] + c[b, a] for a, b in itertools.combinations(range(N_PLAYERS), 2)}


def baseline_scores(N):
    """Simple baseline: how LOW is the capture rate between two players? (higher = more suspicious)"""
    out = {}
    for a, b in itertools.combinations(range(N_PLAYERS), 2):
        opportunities = N[a, b].sum() + N[b, a].sum()
        captures = N[a, b, 1] + N[b, a, 1]
        out[(a, b)] = 1.0 - captures / opportunities if opportunities > 0 else 0.0
    return out


def permutation_maxes(events, rng_np, n_perm):
    """Shuffle each player's decisions across targets; record the biggest pair score each time."""
    arr = np.array(events, dtype=int)
    per_player = []
    for i in range(N_PLAYERS):
        e = arr[arr[:, 0] == i]
        per_player.append((e[:, 1], e[:, 2]))
    maxes = np.zeros(n_perm)
    for k in range(n_perm):
        N = np.zeros((N_PLAYERS, N_PLAYERS, 2))
        for i, (targets, decisions) in enumerate(per_player):
            if len(targets):
                np.add.at(N[i], (targets, rng_np.permutation(decisions)), 1)
        maxes[k] = max(pair_scores(cell_scores(N)).values())
    return maxes


PAIRS = list(itertools.combinations(range(N_PLAYERS), 2))


def wilson(k, n, z=1.96):
    """95% Wilson confidence interval for a proportion k/n."""
    if n == 0:
        return 0.0, 1.0
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return float((c - w) / d), float((c + w) / d)


def build_action_data(game_seqs):
    """Turn per-game move sequences into rows: (i, j, parity of i's move, index of j's last move)."""
    xs, Ms, starts, njs, pids, grps, F_parts = [], [], [], [], [], [], []
    offset = 0
    for g, seq in enumerate(game_seqs):
        if not seq:
            continue
        arr = np.array(seq)
        pl, par = arr[:, 0], arr[:, 1]
        seg = {}
        for j in range(N_PLAYERS):
            pj = par[pl == j]
            seg[j] = (offset, len(pj))
            F_parts.append(pj)
            offset += len(pj)
        idx = {p: np.nonzero(pl == p)[0] for p in range(N_PLAYERS)}
        for i in range(N_PLAYERS):
            for j in range(N_PLAYERS):
                if i == j:
                    continue
                m = np.searchsorted(idx[j], idx[i]) - 1      # j's most recent move before each move of i
                st, nj = seg[j]
                ok = (m >= 0) & (nj >= 2)
                k = int(ok.sum())
                if k == 0:
                    continue
                xs.append(par[idx[i]][ok]); Ms.append(m[ok])
                starts.append(np.full(k, st)); njs.append(np.full(k, nj))
                pids.append(np.full(k, i * N_PLAYERS + j)); grps.append(np.full(k, g * N_PLAYERS ** 2 + i * N_PLAYERS + j))
    if not xs:
        return None
    return (np.concatenate(xs), np.concatenate(Ms), np.concatenate(starts), np.concatenate(njs),
            np.concatenate(pids), np.concatenate(grps), np.concatenate(F_parts))


def action_pair_scores(data):
    """Mutual information (bits) between players' move parities, summed over both directions per pair."""
    x, M, start, nj, pid, grp, F = data
    y = F[start + M]
    n = N_PLAYERS * N_PLAYERS
    counts = np.bincount(pid * 4 + x * 2 + y, minlength=n * 4).reshape(n, 2, 2).astype(float)
    tot = counts.sum(axis=(1, 2), keepdims=True)
    p = counts / np.where(tot > 0, tot, 1)
    px, py = p.sum(2, keepdims=True), p.sum(1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        term = np.where(p > 0, p * np.log2(p / (px * py)), 0.0)
    mi = term.sum(axis=(1, 2)).reshape(N_PLAYERS, N_PLAYERS)
    return np.array([mi[a, b] + mi[b, a] for a, b in PAIRS])


_REFERENCE = {}


def action_reference(n_games, seed):
    """Action-MI scores of N_REFERENCE honest experiments with n_games games each (cached)."""
    key = (n_games, N_REFERENCE)
    if key not in _REFERENCE:
        out = []
        for e in range(N_REFERENCE):
            rng = random.Random(f"honest-reference-{seed}-{n_games}-{e}")
            seqs = []
            for _ in range(n_games):
                sq = []
                play_game(rng, 0.0, "signal", [], sq, None)       # strength 0 = honest play
                seqs.append(sq)
            d = build_action_data(seqs)
            out.append(action_pair_scores(d) if d is not None else np.zeros(len(PAIRS)))
        _REFERENCE[key] = np.array(out)
    return _REFERENCE[key]


def action_test(data, ref):
    """Returns (z-score per pair, p-value for the biggest z) against the honest reference scores."""
    if data is None:
        return np.zeros(len(PAIRS)), 1.0
    obs = action_pair_scores(data)
    mean, sd = ref.mean(axis=0), ref.std(axis=0) + 1e-12
    z_obs = (obs - mean) / sd
    z_ref_max = ((ref - mean) / sd).max(axis=1)
    p_value = (1 + np.sum(z_ref_max >= z_obs.max())) / (1 + len(ref))
    return z_obs, p_value


# ------------------------------ experiments ---------------------------------
def evaluate(strength, n_games, mode, n_exp, n_perm, seed):
    rng_np = np.random.default_rng(seed)
    ref = action_reference(n_games, seed)
    true_pair = tuple(sorted(PARTNERS))
    true_idx = PAIRS.index(true_pair)
    correct_mi = correct_base = correct_act = flagged = flagged_act = unfinished = 0
    rate_pair, rate_others, n_events = [], [], []
    stats = [0, 0]
    for e in range(n_exp):
        rng = random.Random(f"{seed}-{strength}-{n_games}-{e}")
        events, game_seqs = [], []
        for _ in range(n_games):
            seq = []
            winner, _ = play_game(rng, strength, mode, events, seq, stats)
            game_seqs.append(seq)
            unfinished += winner is None
        N = count_matrix(events)
        ps = pair_scores(cell_scores(N))
        bs = baseline_scores(N)
        correct_mi += max(ps, key=ps.get) == true_pair
        correct_base += max(bs, key=bs.get) == true_pair
        maxes = permutation_maxes(events, rng_np, n_perm)
        p_value = (1 + np.sum(maxes >= max(ps.values()))) / (1 + n_perm)
        flagged += p_value < ALPHA

        z_obs, p_act = action_test(build_action_data(game_seqs), ref)
        correct_act += int(np.argmax(z_obs)) == true_idx
        flagged_act += p_act < ALPHA

        a, b = true_pair
        pair_opps = N[a, b].sum() + N[b, a].sum()
        pair_caps = N[a, b, 1] + N[b, a, 1]
        all_opps, all_caps = N.sum(), N[:, :, 1].sum()
        rate_pair.append(pair_caps / pair_opps if pair_opps else np.nan)
        rate_others.append((all_caps - pair_caps) / (all_opps - pair_opps))
        n_events.append(len(events))

    row = {
        "mode": mode,
        "strength": strength,
        "games": n_games,
        "experiments": n_exp,
        "pair_found_MI": correct_mi / n_exp,
        "pair_found_baseline": correct_base / n_exp,
        "pair_found_ACT": correct_act / n_exp,
        "flag_rate": flagged / n_exp,
        "flag_rate_ACT": flagged_act / n_exp,
        "capture_rate_pair": float(np.nanmean(rate_pair)),
        "capture_rate_others": float(np.mean(rate_others)),
        "events_per_experiment": float(np.mean(n_events)),
        "signal_chances_per_game": stats[0] / (n_exp * n_games),
        "signal_used_per_game": stats[1] / (n_exp * n_games),
        "unfinished_games": unfinished,
    }
    for key, k in (("MI", correct_mi), ("baseline", correct_base), ("ACT", correct_act)):
        lo, hi = wilson(k, n_exp)
        row[f"pair_found_{key}_lo"], row[f"pair_found_{key}_hi"] = lo, hi
    return row


def demo_game():
    rng = random.Random(SEED)
    events = []
    winner, turns = play_game(rng, 0.0, MODES[0], events)
    name = COLOURS[winner] if winner is not None else "nobody (hit the turn cap)"
    print(f"Demo game: winner = {name}, turns = {turns}, capture opportunities logged = {len(events)}")


METHODS = [("pair_found_MI", "Detector 1: capture-based MI", "o-"),
           ("pair_found_baseline", "Baseline: capture rate", "s--"),
           ("pair_found_ACT", "Detector 2: action-based MI", "^-")]


def make_plot(rows, modes):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, len(modes), figsize=(5.8 * len(modes), 8), squeeze=False)
    biggest = int(max(r["games"] for r in rows))
    for c, mode in enumerate(modes):
        mrows = [r for r in rows if r["mode"] == mode]
        sel = sorted([r for r in mrows if r["games"] == biggest], key=lambda r: r["strength"])
        ax = axes[0][c]
        xs = [r["strength"] for r in sel]
        for key, label, style in METHODS:
            ys = np.array([r[key] for r in sel])
            lo = np.array([r[key + "_lo"] for r in sel]); hi = np.array([r[key + "_hi"] for r in sel])
            ax.errorbar(xs, ys, yerr=[np.clip(ys - lo, 0, None), np.clip(hi - ys, 0, None)], fmt=style, capsize=2, label=label)
        ax.axhline(1 / 6, color="grey", linestyle=":", label="Chance (1 in 6)")
        ax.set_title(f"Mode '{mode}': pair identified ({biggest} games)")
        ax.set_xlabel("Collusion strength"); ax.set_ylabel("Fraction naming the true pair")
        ax.set_ylim(0, 1.05); ax.legend(fontsize=8)

        key = "pair_found_ACT" if mode == "signal" else "pair_found_MI"
        ax = axes[1][c]
        for s_ in sorted({r["strength"] for r in mrows if r["strength"] > 0}):
            sub = sorted([r for r in mrows if r["strength"] == s_], key=lambda r: r["games"])
            ax.plot([r["games"] for r in sub], [r[key] for r in sub], "o-", label=f"strength {s_}")
        ax.axhline(1 / 6, color="grey", linestyle=":")
        ax.set_title(f"Sample size effect ({'action' if mode == 'signal' else 'capture'}-based MI)")
        ax.set_xlabel("Games observed per experiment"); ax.set_ylabel("Fraction naming the true pair")
        ax.set_ylim(0, 1.05); ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig("results.png", dpi=150)
    print("Saved results.png")


def threshold_table(rows, modes, level=0.8):
    """For each mode and sample size: smallest collusion probability where the true pair is found often enough."""
    print(f"\nSmallest collusion probability where the true pair is named in at least {level:.0%} of experiments")
    print("(a rough guide only: with 100 experiments each rate has about +-10 points of sampling error)")
    for mode in modes:
        print(f"  mode = {mode}")
        for g in sorted({r["games"] for r in rows}):
            sub = sorted([r for r in rows if r["mode"] == mode and r["games"] == g and r["strength"] > 0],
                         key=lambda r: r["strength"])

            def first(key):
                for r in sub:
                    if r[key] >= level:
                        return str(r["strength"])
                return "not reached"

            print(f"    {g:>2} games: capture-MI = {first('pair_found_MI'):<12} baseline = {first('pair_found_baseline'):<12} "
                  f"action-MI = {first('pair_found_ACT')}")


def main():
    self_test()
    a, b = PARTNERS
    print(f"Colluding pair: {COLOURS[a]} (seat {a}) and {COLOURS[b]} (seat {b}); modes = {MODES}")
    demo_game()
    rows, control = [], {}
    for mode in MODES:
        for strength in STRENGTHS:
            for n_games in GAMES_LIST:
                if strength == 0.0 and n_games in control:
                    r = dict(control[n_games]); r["mode"] = mode       # no collusion: same control for every mode
                else:
                    n_exp = CONTROL_EXPERIMENTS if strength == 0.0 else EXPERIMENTS
                    r = evaluate(strength, n_games, mode, n_exp, N_PERM, SEED)
                    if strength == 0.0:
                        control[n_games] = r
                rows.append(r)
                print(f"[{mode}] strength={r['strength']:<4} games={r['games']:<3} "
                      f"pair found: captureMI={r['pair_found_MI']:.2f} baseline={r['pair_found_baseline']:.2f} "
                      f"actionMI={r['pair_found_ACT']:.2f} | flagged: capture={r['flag_rate']:.2f} action={r['flag_rate_ACT']:.2f} | "
                      f"capture rate pair={r['capture_rate_pair']:.2f} vs others={r['capture_rate_others']:.2f}"
                      f"{'  UNFINISHED GAMES: ' + str(r['unfinished_games']) if r['unfinished_games'] else ''}", flush=True)
    with open("results.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("Saved results.csv")
    make_plot(rows, MODES)
    threshold_table(rows, MODES)
    print("\nRead strength = 0.0 as your NEGATIVE control: 'flagged' should be about "
          f"{ALPHA:.2f} and 'pair found' about 0.17 (chance), for BOTH detectors.")


if __name__ == "__main__":
    main()


## Optional: download your results to your laptop

In [ ]:
from google.colab import files
files.download('results.csv')
files.download('results.png')
